# Agentic CONTROL — the base model through the identical scaffold

**Runtime: A100 or L4, ~1-1.5h, checkpointed every 20 problems (resumable).**

This fills the missing 2x2 cell. Notebook 20 ran SFT v2 through best-of-10 +
self-repair and got 67.7% verified-resolve. But a scaffold flatters *any*
model — so 67.7 is uninterpretable until we run the **untrained base** through
the byte-identical pipeline. Everything here matches nb 20 exactly (same 164
problems, same verifier, same best-of-10 + 2 repair rounds, same native prompt,
same temperatures) — the ONLY change is the model.

**Registered prediction (S2.37, BEFORE running):**
- base + agent lands **well below** SFT + agent (67.7), predicted band **30-42%**.
  Reason: best-of-n can only *select* a correct fix the model *generates*; the
  verifier can't invent one. The base rarely generates a correct fix (frozen
  pass@10 was 23.5), so best-of-n has little to grab. Training raises the
  ceiling the agent harvests.
- If base + agent surprises us near ~65, that would flip the story (scaffold is
  everything, training irrelevant) — a bombshell we'd need to know.

The 2x2 this completes:

|            | single-shot | + agent |
|------------|-------------|---------|
| Base 1.5B  | 17.6%       | **this notebook** |
| SFT v2     | 34.5%       | 67.7%   |

In [ ]:
# 1) Setup
from google.colab import drive
drive.mount('/content/drive')
import os, sys, json, time
PHASE8 = '/content/drive/MyDrive/rl-post-training/phase8'
BASE_ID = 'unsloth/Qwen2.5-Coder-1.5B-Instruct'
CKPT = f'{PHASE8}/agentic_base.json'
!rm -rf /content/ptl
!git clone -q https://github.com/nidhi1603/post-training-lab.git /content/ptl
sys.path.insert(0, '/content/ptl/src')
from prompts import build_repair_prompt, extract_code
ver = !git -C /content/ptl log --oneline -1
print('FACTORY VERSION:', ver[0])
from datasets import load_dataset
probs = list(load_dataset('bigcode/humanevalpack', 'python', split='test'))
assert len(probs) == 164
state = json.load(open(CKPT)) if os.path.exists(CKPT) else {}
print(len(probs), 'problems |', len(state), 'already in checkpoint')

In [ ]:
%%capture
!pip install unsloth
!pip uninstall -y -q torchao torchaudio torchvision timm

In [ ]:
# 2) Verifier + CALIBRATION GATE (identical to nb 20 — run void if it fails)
import subprocess, tempfile
from concurrent.futures import ThreadPoolExecutor
STD_IMPORTS = ('from typing import List, Tuple, Dict, Set, Optional, Any\n'
               'import math, re, collections, itertools, functools, string\n')

def run_candidate(code_str, rec, timeout=10):
    prog = (STD_IMPORTS + code_str + '\n' + rec['test']
            + f"\ncheck({rec['entry_point']})\n")
    with tempfile.NamedTemporaryFile('w', suffix='.py', delete=False) as f:
        f.write(prog); path = f.name
    try:
        r = subprocess.run([sys.executable, path], capture_output=True,
                           timeout=timeout)
        err = r.stderr.decode(errors='ignore')[-400:]
        return (r.returncode == 0, err if r.returncode != 0 else '')
    except subprocess.TimeoutExpired:
        return (False, 'TIMEOUT')
    finally:
        os.unlink(path)

def verify_many(codes, rec):
    with ThreadPoolExecutor(max_workers=8) as pool:
        return list(pool.map(lambda c: run_candidate(c, rec), codes))

gold_ok = buggy_fail = 0
for rec in probs:
    ok, _ = run_candidate(rec['declaration'] + rec['canonical_solution'], rec)
    gold_ok += ok
    ok, _ = run_candidate(rec['declaration'] + rec['buggy_solution'], rec)
    buggy_fail += (not ok)
print(f'CALIBRATION: gold pass {gold_ok}/164 (need >=160) | buggy fail {buggy_fail}/164 (need >=140)')
assert gold_ok >= 160, 'Runner plumbing broken — run VOID.'
assert buggy_fail >= 140, 'Tests not detecting bugs — run VOID.'
print('calibration gate PASSED')

In [ ]:
# 3) BASE model + generation helpers (only change from nb 20 is the model)
import torch
from unsloth import FastLanguageModel
model, tok = FastLanguageModel.from_pretrained(
    BASE_ID, max_seq_length=2048, load_in_4bit=True, dtype=None)
FastLanguageModel.for_inference(model)

def gen(prompt_text, k, temp=0.8, max_new=768):
    text = tok.apply_chat_template(
        [{'role': 'user', 'content': prompt_text}],
        tokenize=False, add_generation_prompt=True)
    enc = tok([text], return_tensors='pt', padding=True,
              padding_side='left').to('cuda')
    with torch.no_grad():
        out = model.generate(**enc, do_sample=True, temperature=temp,
                             top_p=0.95, num_return_sequences=k,
                             max_new_tokens=max_new,
                             pad_token_id=tok.eos_token_id)
    return [extract_code(t) for t in tok.batch_decode(
        out[:, enc['input_ids'].shape[1]:], skip_special_tokens=True)]

def base_prompt(rec):
    buggy = rec['declaration'] + rec['buggy_solution']
    return build_repair_prompt(buggy, [rec['test'] + f"\ncheck({rec['entry_point']})"])

REPAIR_TMPL = """{base}

A previous attempt:
```python
{prev}
```
It failed with:
```
{err}
```
Return only the corrected complete function in a Python code block."""
print('base model ready')

In [ ]:
# 4) STAGE A — best-of-10 with test-verified selection (checkpointed)
N_INIT = 10
t0 = time.time()
for i, rec in enumerate(probs):
    tid = rec['task_id']
    if tid in state:
        continue
    codes = gen(base_prompt(rec), N_INIT)
    results = verify_many(codes, rec)
    n_pass = sum(ok for ok, _ in results)
    entry = {'n_pass_initial': n_pass,
             'first_sample_pass': bool(results[0][0]),
             'solved_stage': 'A' if n_pass > 0 else None}
    if n_pass > 0:
        entry['final_code'] = codes[[ok for ok, _ in results].index(True)]
    else:
        entry['fail_code'] = codes[0]
        entry['fail_err'] = results[0][1]
    state[tid] = entry
    if (i + 1) % 20 == 0 or i == len(probs) - 1:
        json.dump(state, open(CKPT, 'w'))
        done = len(state)
        solved = sum(1 for v in state.values() if v['solved_stage'])
        print(f'{done}/164  solved so far {solved} ({solved/done*100:.1f}%)  '
              f'[{(time.time()-t0)/60:.0f} min]')
json.dump(state, open(CKPT, 'w'))
sA = sum(1 for v in state.values() if v['solved_stage'] == 'A')
print(f'\nSTAGE A (base): {sA}/164 = {sA/164*100:.1f}% verified resolve')

In [ ]:
# 5) STAGE B — self-repair with execution feedback (2 rounds, k=4 each)
K_REPAIR, ROUNDS = 4, 2
for rnd in range(1, ROUNDS + 1):
    tag = f'R{rnd}'
    todo = [r for r in probs if state[r['task_id']]['solved_stage'] is None
            and f'tried_{tag}' not in state[r['task_id']]]
    print(f'--- round {rnd}: {len(todo)} unsolved ---')
    for i, rec in enumerate(todo):
        tid = rec['task_id']
        e = state[tid]
        prompt = REPAIR_TMPL.format(base=base_prompt(rec),
                                    prev=e['fail_code'], err=e['fail_err'])
        codes = gen(prompt, K_REPAIR)
        results = verify_many(codes, rec)
        e[f'tried_{tag}'] = True
        if any(ok for ok, _ in results):
            e['solved_stage'] = tag
            e['final_code'] = codes[[ok for ok, _ in results].index(True)]
        else:
            e['fail_code'] = codes[0]
            e['fail_err'] = results[0][1]
        if (i + 1) % 20 == 0:
            json.dump(state, open(CKPT, 'w'))
            print(f'  {i+1}/{len(todo)}')
    json.dump(state, open(CKPT, 'w'))
    s = sum(1 for v in state.values() if v['solved_stage'])
    print(f'after round {rnd}: {s}/164 = {s/164*100:.1f}%')

In [ ]:
# 6) THE 2x2 — training effect vs scaffold effect, cleanly separated
tot = len(state)
sA = sum(1 for v in state.values() if v['solved_stage'] == 'A')
s1 = sum(1 for v in state.values() if v['solved_stage'] == 'R1')
s2 = sum(1 for v in state.values() if v['solved_stage'] == 'R2')
first = sum(1 for v in state.values() if v['first_sample_pass'])
base_agent = (sA + s1 + s2) / tot * 100
# the three measured anchors:
BASE_SS, SFT_SS, SFT_AGENT = 17.59, 34.51, 67.7
print('=' * 60)
print('AGENTIC 2x2 (verified-resolve; single-shot = frozen pass@1)')
print('=' * 60)
print(f"{'':<14}{'single-shot':>14}{'+ agent':>14}")
print(f"{'Base 1.5B':<14}{BASE_SS:>13.1f}%{base_agent:>13.1f}%")
print(f"{'SFT v2':<14}{SFT_SS:>13.1f}%{SFT_AGENT:>13.1f}%")
print('-' * 60)
print(f"base single-sample native (temp 0.8): {first/tot*100:.1f}%")
print(f"base Stage A (best-of-10): {sA/tot*100:.1f}%")
print('-' * 60)
print('EFFECTS:')
print(f"  scaffold lift on base : {base_agent-BASE_SS:+.1f} pts")
print(f"  scaffold lift on SFT  : {SFT_AGENT-SFT_SS:+.1f} pts")
print(f"  training lift, single : {SFT_SS-BASE_SS:+.1f} pts")
print(f"  training lift, +agent : {SFT_AGENT-base_agent:+.1f} pts")
print('-' * 60)
print('Registered (S2.37): base+agent predicted 30-42%, well below 67.7.')
print('If base+agent << SFT+agent -> 67.7 reflects a better MODEL, not just')
print('the scaffold. If ~equal -> the scaffold was doing the work. Read it.')
print('\nBring the whole printout back.')

## Bring back to the session
The **2x2 table** and the four effect sizes. This is the honest completion of
the agentic track — it says whether 67.7 measures the model or the wrapper.